In [5]:
import sys, os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

def load_data():
    X, y = fetch_california_housing(return_X_y=True)
    X = (X - X.mean(0)) / X.std(0)
    y = y.reshape(-1, 1)
    return train_test_split(
        torch.FloatTensor(X),
        torch.FloatTensor(y),
        test_size=0.2,
        random_state=42
    )

def build_model():
    return nn.Sequential(
        nn.Linear(8, 64),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

def train(model, X, y):
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    lFN = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True)

    model.train()
    for _ in range(500):
        for xb, yb in loader:
            optimizer.zero_grad()
            l = lFN(model(xb), yb)
            l.backward()
            optimizer.step()

def eval(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X).cpu().numpy()
    y_np = y.cpu().numpy()
    return {
        "mse": mean_squared_error(y_np, pred),
        "r2": r2_score(y_np, pred),
    }

def save(model, metrics):
    os.makedirs("outputFile", exist_ok=True)
    torch.save(model.state_dict(), "outputFile/cal_housing_mlp.pt")
    with open("outputFile/cal_housing_metrics.txt", "w") as f:
        f.write(str(metrics))

if __name__ == "__main__":
    X_train, X_val, y_train, y_val = load_data()

    model = build_model()
    train(model, X_train, y_train)

    val_metrics = eval(model, X_val, y_val)
    save(model, val_metrics)

    print("torch:", val_metrics)

    success = val_metrics["r2"] > 0.60
    if success:
        print("pass")
        sys.exit(0)
    else:
        print("fail")
        sys.exit(1)

torch: {'mse': 0.25187867879867554, 'r2': 0.807786226272583}
pass


SystemExit: 0

/Users/Varun/Library/Python/3.9/lib/python/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
